# Final Project 16: Building Footprint Extraction

Notebook cuối kỳ cho bài toán `Building Footprint Extraction from Aerial Images`.

Workflow chính:
1. Chuẩn bị `Inria Aerial Image Labeling Dataset` và split ở cấp original image.
2. Rerun baseline classical: `K-Means` và `Otsu`.
3. Train/tune supervised baseline: `Linear SVM`.
4. Train/tune deep learning model: lightweight `U-Net`.
5. Xuất `final_summary.csv`, `per_patch_metrics.csv`, model checkpoints, charts, qualitative overlays và LaTeX snippets cho report.

## 0. Kaggle Setup

Trên Kaggle, hãy bật GPU và attach dataset chứa `AerialImageDataset/train/images` + `AerialImageDataset/train/gt`. Notebook này có thể chạy theo hai cách: nếu bạn upload cả project folder thì nó dùng `src/building_footprint_final.py`; nếu bạn chỉ upload riêng `.ipynb`, cell dưới sẽ tự tạo module cần thiết trong `/kaggle/working/src/`.


In [ ]:
from pathlib import Path
import base64
import sys
import zlib

PROJECT_ROOT = None
candidates = [Path.cwd()]
input_root = Path('/kaggle/input')
if input_root.exists():
    for child in input_root.iterdir():
        if child.is_dir():
            candidates.append(child)
            for grandchild in child.iterdir():
                if grandchild.is_dir():
                    candidates.append(grandchild)

for candidate in candidates:
    if (candidate / 'src' / 'building_footprint_final.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    src_dir = PROJECT_ROOT / 'src'
    src_dir.mkdir(parents=True, exist_ok=True)
    embedded = """eNrNff1327aS6O/+K/j09rVkIiuWnLSpztU966ZOm918HTvt7j2qDg8tUTKvKVIlqdi6Xu/f/uYDHwOS+nDSve/ltJYEDAbAYDAYDAaDeZEvvTCcr6t1EYehlyxXeVF5UZblVVQleVYeHem0YrGKijLWv/9e5pn+voyqa/09L/W3Ispm+VL/qpJlfDTH+mZRFU3TqCzj0lRYzpJp1bVZDLkCvGlypaE+YjWUUW1WSbbQ6WfZpuu9itI0ukpj0+Dp54Fpa34FaERrV2leQUpvtcFvUL23Siudn62Xqw2mZSudtIKuQALCzbgF5U0aR0XWWyWrOE2y2DPIb+JQJ9ZAi3hV5NO4LEXbLytEXcwup1EaF26B8vNSg70FbFFx+duro6Oq2AyPPPjHlPhjtuxF6yrXkJhwFN9N41Xlvctn6zR+n1ev83U2Oy+KvOCis3hOgH5SxQVSbfQ+z+Ku9wRGuYSPJze3+C0Yet7/RmLHQy9ZZHkRj7P8uIih+ITw4L8iBubJPI3JS+biOxAR6IvIvTgtY288OTo6evPu7Ofz8PLX16/f/Of5pTfy7ju9Kpl3uh598pdVtqDPv6/0Z7zoPBy9O/908eZV+OrD21/fvceyY2pIJ8nXnS5/BVaK9Xeg+TQpgZF1AvwGRtG/5n39LQsBdGZ/LSr9fQq0q8K4KHRCVMRRWOD0cMqIZFuYEqOrEsvniAH7/6+Wzemvd363iguYH1n1Ks/myYIHiedPWMbxDMifVdDZ5wPKgXkxvQ7L5B+xznjRd3KqIpk18qoiSjJu4dCbp3mEmSe9708o93OUNvP6L7hkXFatmZS7jO5Cxk21xyVX/F887COvPzg5MYBYTTvYqYCiCreDEdzNMo6yMpym6xLYrTQkkpnIiDqjfyJz4lVp+9LvKZwwHNcxoExWYZosk8qCDHonAqJK0jhcAJGHXrVepfEYquhiPRMA9V92vZcBQedVuQ63I+33mL7LvFhdA0RexuGNHNYfRHa+ijM3VxUGwl+tk3QGUiVEftPZA02pVXIHky8EHpPjZOgih0flUwGBJzxRMCCSwqnTc+pL1+v1etR3YIsu8MaLLhI1MGUQtxyMFyeiUgQQfBHOc2joOoP+NDmJe7TOYpiSq3x6LVppc64a8+O5zSTZirQCho7FWMTHpxboNk4W11U4A2mxcWAEImhuEmfT2nBQFiwh4W1e3Ai+HNjc6rqIy+s8ne0m5CkS8rmi5knvO/zzfWCoRqgaZNOoWyhn6o9Wq3QTElflab6ADl7leQpQn4p1zAT+Yx2lCaoAn+MwvouW0EDTk5fM2utqtQYKJTCmIG8gvcNJZYeXGJBxJVRW5HnFEKYx+MGTCSRcAoAxjBiwPuAqNTWggKWFWWs6z26ixSKNnyUZ1PRM1VE+K6NFVPRPTmBIr/MZ5BZJdByBTI3S42QZLeJj0A1wSV4cqzLPzij3DWb+xElKZrfU80/E9yg8DobDGvLT0FJtN+TPw+0AwZFRIxQfoKrml3E6D7zjv5KmNqxrCB81SM/yDiA6QizIKYs0vwJ2xvXON4seoUOGkWtiz8DwdMhWvfaMvOzF2eekyLNx5+PfPv3y4f0vZ5e/XJ6f/9RBxgImU8DcijgrUQ+2zSv96RyEUH19pkahxspsih2bqPYBtwNiKNWThAn04nxdorpjiYzwnSEVE6RfxrCAT0uV4T2zKQII2gGNlUA6RWLKZyDJJSJOECCglYLeKGF0isQTZckcFmUHlUljwAfWSvOCOgqDxx3ugYxax6UfWIbA9N7yBgjsw54CyFqOUPR0vfgugYU/v6GfTDTFPIRKDVOIew+QzfNonVY+oR/iJoCGBT65ItRDQdMuQcGexgzFQxU0OBP5gACC7SWBxxZxBmwwbZYnkB6sb0s10rCKgrL7CRRnUrr9eefD1d/jKeyO5qROe/f4V9XZC8MsWsL260Hry/92+eE9TAmcfMk/UJHumHkSgUCeEhPuYE07XzTLtXM2N/Y2gdHyCXLMDDnB4Y0NZlVhD8neCXqoiPidW9DKYfXLUe0YddbV/PhlJ8Ad0tySBwv0ZrCd8nmDR3V2vTmqSjNAPBp0PTWMI2dQdXeBm/MUeiwXkx39tmLHrCywoKTAU2Oao7hdmOghxjnqLFKm2bZwD5bKOJv5JLrqBQLDLSBmFnEFksbvvHl/8eYs/Ons09nl+afw4sOHT51gL2YpploQTFRNonR8V5nSq4AnHc44bGPLqsrlee0Iae0AUlBhd0XpmC5J2B5NS2cGY4WsKwJ6rNiBL1CU+x0CkN1XuJHFTeFeUuKHXwOj/uZZlWTr2MlQEtaWZwlSr8PXUoobgd9omSyBgXV3PCBUGyBs34KWPm8fROKGIyP+CCMOhmXCo//5pilZhGXUZuY6nt7EM9TMfs9gA50nGQgizzv27lcPHcEzpplSeL1OapYDoYW9Qf1Fa3jc3duIRdcc4XveJaTX54sHFTa5uwdts2zVecWNHv6e3av2P3S0ykEiASaaEQcks3zWMEmyb1+XmTh7V141EHaRa4yRsyiWN+2wxgDwoFo9TapNiAYbEvU+7FiXpBRTi+FTSS2AwuFSgzW9pkGa0lqKRfTkmV7DrJkli6TyA2eVRAQ93P2v/E54DPIYinfW2U2W3wKVuSnYaqIcbRioVz79pUQmZJehSLk3hMUvSonn5pa4apXr+Ty5g1brEtB/i66H3dUyRcC38LBeiy2QmVCqDqCCazpqEaxuQ+ade9EWJOHDPSN76MgpaYrvnlsGTNKcdjNM2llSTvPPsaIqDXXJtC0tLVtpS2sUDJxiVyo5tKl24UJy2C4RY4CuBkqvrQYVkYJEauDIHUkJHoQ0v40BjHhqB3FbZXGyj5NsPwNpB3TRUke1GK2xjVmLsChTxDILyaiLdYZmZa1jvc+5j8+wYlB9YOFjeYSdu7cEeiDpeq+b99Bx5hDVpNWtFeyCndGsj0zXO2SHYMAnQ9kptJvDTCbDD6kXwkTnPSUBasxy6rc1xZGFpeuhcbHK01E/Pv4uqBPoN1QwmTwdF7fEa3F6y3VZeTHu/hF7Ry1q5TXwS0pLieI2IoMiW7aAdLX/uqAP6oswXxrAnsLka4yqggy6UEVogkhBt7SZKg9bDnmwHQRhD+Ppa/gnXo1qgS7zmbBtK2F6b3U4Xc1TVfivulGWphrrMrrzgfIa5bEuG1gzK1JK9WI8VNkTbWeVmbraYb0BE2N5bQPXrRxOnOVNLUFDbkTX6wAM/IK/8B1xYRZ86GUJMlhTBMmFS04VEseXPn8MW1m4tsWAAc4koJJX93Y7SMi6PLFIYhFy2i81lEoEQpjafNeDpHKpzoZixjx/VsLOCc+utFgg+4V3j2Vh5oOoiQpqxlUOAvQeUY0xb6LEArVOywT9z0LhaCCEomCaR7OwWFz5dvEk+sB2Ec91ikjvRqkRIzyV6iXLIo5mPu46sRTsiDD1zbuL87Of8GTjw4Xdh1IxEJ+u6NymooH+lK/TGQkXrIPLD0Hhg3pqcg7rnH4GeZVCQYLjdlADwh9/vhhc/PxjILuJAnNfP0n0HtDNny/O/nb56uztuekqlfzynmLx9o5ijuoG2e1XeZnQ2aZ/TVZesjN1YRs8q67V99oBCzG1PlGx63XN9G9lO+P1/iLwoCZGNTipDe1HrfKmicP2mlx9YIPcDOIWll8QS6ryY1n5U6+v+1Cbb3dOWW7hAUWdRur12990vTtXITUwagDooIIt1mERT/NiVvK+gldanGGkF3ftpAkbS65jX6Y9wLCu9TNM69LcNRb0lgOm7pEdXovyLNtMzEaCGj1sBWlT07TIkn2xSpTV5EZuj8Z6vzEx+rRRpfU80wX36WE1xLxtmQStqlxzBu6ehe8SPs6m2Yu9vq9pcQ9aidghBW2RrbJwu2wxTd8pYNrkKc7JdqnzKMmDiuZQ9vuhq5RLKYxkA0ANilbxeDiYeP+Lh8+mtDVCqHF6OcOlitpOBb1lUi75UG8n/Vk26Hk+qrfGkQw4mZFxW2WmQkHab89KC+d3m8xQ00dLjMYSft9I4WN8wpjMQH2BLZ2VFQ/hPVmx7AQJeJcX3m/gf9jlddsREgrAZlFtAbSYAdr+2AKN+2+Aq+32W5sYbEFxB+XvtuQh7k13F4lwELAB7qjs6hoySodkrjMN24sYrlYlxNxrFHhwUhQTkqgxktfxD0GORu1fsUgAGrgU0nal3L/fgEXLqsL9vtGDB6dWDT79/mFsR19IQ1U/1oEbFjr71I3qyia1FVHbIwN+E29GabS8mkUeyAi/GEuGgs0jJGzU552x82pLHiNRKyd6D0VFHJpTlx12cFYX6ktiXZmvLV56hasfxbXu5cTh1r7jBccEONpq0g+aqzpAN619jgX+qK4m4DLaZoTZsrbuXht5n6IV/qZRh7+TzAv03m7XdsppbpHflq6+8Gdtkwx6LWXvXXHXrUk1kmc7ZZeQWg/qKHPWwxPh1wXC2BqDXpWH0/KzL/ljLE4KJ8aSyqQx7NyDUh0+F7obvY7SMg6sd9HVhqHds1M9wZFBGz5IwkzLc19DCY8JAaOEgkEl/I/k4aaaku0NopRhq4orh7bGc8Q+XaebLJeESNrPHaKNUZqGViCN+SufS6jUJGt0xBzSCkABN2kOuqjmsFHfP961ru5qbmNSOG17TLu0PvGwnxVrh6+OsJZVdBvtNceZkeIM1US5b0EhLKS33Vh3xSZ74loTjPWBpinjHMuFXS8pSnW223ingF3WNfydNp4piDstEze1jI3OEFtGF0IoJhJUa5/jjQdKDazYUou8g7Q7J405cFGp7c6jSlkXgilorAs9GldJFhUbJgdbDyydW20bwp4Aykl/8L1YmtmxVw2u8q+TQ7h9sa7XkkZXapNjrDOq5dY6c/Hzj4O3Zz8G1iVRFwEWq+JXb89+OffR4/AtOhyO6FS45oXYJcX+Z9DRL4FQAsT4Nga6OeNh14P/TnB/SzA98iDzZdZ20xJAyaZDs6VhqYwX5GLAbplfR7nr8vNBlPvl8regF5XkggE4yPHudKB6sEDqU7FtINN8CaxD1nDIAnkwBZJn8L8/xtLPvMGLF2iZx+aoH6BoRHdJOVIY6nuo6yjL4hTFtcbNezLhvSnz0MUPcv3jvi27tUPTAo+Dksjxo0N6fDq/eBe+ungDn2/OwvOPl3jC0Eh/d/afIX61ayVyinCvbc2IV6V2FMO/Ydcj/zZc7GAI46JU46RG3Upx6msrSu3nazPZXmNAVT9tSlTBIrGqytGpTZun0aIcYdX//u787P1l+PFj+Or8PXTwUjaYW4sCk74YijsDp+hrOgSUj0piR18lMoRxziVkSjoicLHAkwQFizPpRW0HAEC313ER+7o9oxqyLrIXTD85+mtA/7I+vdAR+Z83udRqCX2gSyRQDkorUdGfPEJotfpPHyq5nAZIuWUzDHsK05JxoRVwQGJFapohv1ycX/4S/vjm/dnF38I373/TM4fTP3y6/HWbAdo43SZT0EOnKbD2etVYfR4zLsprPC4y4i1sB2yaLqtiPYXKgVPO0xhR+Jjz7sPFx1/Ci/NXn7oebewanudswWkkK75kH/Svqsu6scuqbKr2dULSkIxFXNZX+fyOqNXgdx4ZrvPV2w+X512HMnuRqiyJ5sPH8/dd2WVzWKmFmf4sq6iCj1Dzcw4ieVrFs1f5cgVCKqvK/0iq60uEEhUxVPIZNlyjl3JbzbLkH3GRl8DxN7EuYzVkliTGct/vmka5R/7UsLESFTRPX4WXn84+hWcX52cT76/sqdq8QeBsKLlRYyuD6BsqAzAjmjqw9nihazMabdmuYcHQDWs03UFE37eqV9DGAS0U1beTgCnNsa3qybHXD4znL+5/cOeOxuw09vFijzsjF1Wj9VZ3pxVX6eZYNEQlQuqXmBgoFbaRuajUyfHKLg5aRESwf2eMXSoK2/D1Unv9zPcU+O9miay9xH9vrYMy4Lcqx3ASAErYbCwuM9cZi99albh4tFXIm5t8zXdykIMVgpF3whfIiNI+UOoZZymzUTKNbRnT5KemcbXyA++JRzj8BqztNl8ca0FL2NBc2cDui+bh36cwREFATn/6F7CvAj7RF2T4TtrXVpQ5FWVtFc37UImlgO3jE90GpohKfapSFeKWDKcOLR2x+aSuugLATgG8XdcCoKdB7X6dabIhyzNPcyHuNp94/RPTQ+cSnimpyPfMY/ZslGp6B+J1wiFyorAR0b1C2rELtVPcMBxagjru9XThcKhIJh34+3ic0Rcp6hKiunDk88/AAVhUIhsoJjLtRUUNEV2VCgk5qQC4hK/fYxzWKd8OSy1wftfh7H1H2ZD6sB67SHTTtGtKtFgU8YJE8nq5BFnp4/013trP5kPH4EPSWCboyxi3Bx3ULmO8NQSiqMjX5BEra+pR6tXG7zBYp0tGfmUYEscS+W3dmuNYB/XFDkQx1FW6ucZ42aEjaR9PQ6j2oHa80iljWOtmJTsgGSoT7FhndiZKqm4pazq5vTzuvBwED45Jki+lIMHcW7m1I7/8djzv3DPwQ4g46fKNrJMzTYW7ypfVbGtxyPNns3w+OhE4pCUcvrvuCY7VEA3YdX2AB4p3qPy97qSw0yOga9ZQSM+LobmmPhYGvolj4dvrugCbk1uYQjnMj7J07u4p74Uvmwl0aRvaiM71eDXcnGHN4nI6mnfOFUm8e0GHhw4eK5V0hRxP8py2WX3VGpR1Lc5lIDbL0eFQzUBqhxFU2YIc/4EePSDLPCR5h/6rFhUvF4bcvEMVOJiz27GAQKI6rLsB37nC460W7RBb3M5kR7sPtOtCYMt5sjzzdk2pkDRpgddnPBqYf7dBOqc/rtFYnUsebTvZ1tD0sw3uTskuaThuAdvUwDbtYFoUDfXQNWGePFHj5GZZSXXghL+rimha8dXncB7j1j9usUe2G4Z3Gg21EfDR1t1duP40g6e1zJxMvGegg37/g2qqtNlgju2EzRmIHN4NAFEOahP65wT7qFXmsFML7xTCS/zlL9hAgsh+C08Hr7vooAb7OrIejE4DUXCzp+AJlXUKAhCekCx4D17+UVS+bsQT05ynBr9O27SVRpOVb5Keec/RKnyCf+huPPNDjvshXPdUW6/SdaGa6v/Q9X5QKxnDlX+0gkIztpaoZqIr8Am74GQJeoGL8Fg25In4QQ3WOzM9LRjjDMTl9MaKu7G7bC+u3BmJ5wBOAnCRm4B9UHyFtt1JI5cIuR3CtnofDFClHWRy1BAZutOO2d0ksu/UcX/SOsH0pU30IVFRFfhsndc4h6iOucEc2jVTRTQHG8ehqz3sh+Iy9M94XxUXW6UXHHCwaAzfbGWzXQ62mV20vyosS3fMF/M0qrI8Q/uVMFz31d4vXuyDPLG+QqgBK9y819ROQgoNJUoHfQDGE8Ik82XRboNo3jMQM4EoCAhFQY2+peAx12LLTq/zMtY1o8sQJMAWUVcO2wUULwSAB8OrNJrG8sRb4OBGCByqHQYH/N6Do+VQyjawKypSJ3Zx2kLmYRPp1o7tp3XQ2mS0GgwPGAC02g1OTtqH8DQIdjb1EPo17qYwolYZoGg50aZf/dvxbiY/FDSlLiNQTO783RsEb58b1875ys5wdsqr+9MhpLc7bqH3WlHpxgh8diOw2Q/i6vQt24XOOetT3uVv79SNGE3DTvBo3V+drmvlvuWwXThh1RaqbYqdRmkL3YUsqLtAAf6Grl/bZLfGhK1Xbq+tYXa6OEayEiKu3ivoOi3AxgXYGAA1fPp6EYzLZ16AFcqtGt5GlpGSYdMsKGS6kguqNMqFmqMVBwkyDb+J41Vt9uEZJx1QSESBmo1t2LYKN9tv9W2M1U0E1XT+ppGv5rAq2NUQ+koIVV9+Xn7RPIXkUAWgIJqLeUtR6eTBQL0B3mi3wBA8TVE7yPVFRJezmpcbRM4XOykTN87dlb4aqWa7yhFFI1Oxj0adqyjFABizmg/1bB2lPDxuug7xRAMrYz51ax7tLJEq4MBRTURZyECexu/b+hN1evOk8uskdo9hEay7c+tv/Hlw9Cku1VaHHsI25NiDh5/WPkqSPVqKGe2NKaL6YURVYPQ53qoav3997/8WFT/l8lCXCbArgH1ZjaBth9mARvVBTbB1FtP8kkHo9tvM6IrmfrAtdrL6PJT7fndWcqAx4TEpKpahQnYHKmu/4bwL93h4ANrJ0cHmuyt0WbVMKcNsUdZUhSWyEdzGJ6KkOkE77qudNwUeYBmh44qI2G/WRqbmFAtRa14zQtUZb+XrqvBa6Y6dn82loU2aXW1wCFAmwlfhvSpfv9zhUru++SQffDXDliNFqC0zHSCYhWvicb7o1sxK4vCRDdLclTEfFjUN2q5b9it0OVaCmPyUQ3nGpDycQ2sEc34/OPeKqBF/tWPp2uDlEONHM9OsMPjZzCbu4ZYePfYCAEdf5ThAtq6uV/PK5cBU5Cqc0qpFo2L8EuZ5Xq0KEEU9xqeuValBJ+5pMe45IFvcgVVoLXZShioZfpuDspJ9siOmgq4ilo5ThaQO07xUiiNKyqr0+UMtHbDsLGL1qxa/SsVzzYvp9ZFePK5odmFSr0wWyzyZKXT6HHxJm/d+1xt0PWXSSsi/DB0j6DRblV4vfcb3RLehS8UVojjLl01gBYIxA0xGW2nmNH/QOwH0Tv1PyfaF58xcxVMbJ1JRFg/BjwmFmj3O5ooCGBLd/R3kqif0sowC+mYmzGdZej/l66s0hhXjs59lPY6VK/YmNIIhKGZVGFLYOmSGUPtbqnvBwEpuUu1yvDl5WK9Q0egZhPXr5em8d5XmU3RCg8Zcxn+sYTVLorR5RQ6ysc2DmS+a47YE7x2hw1LIlk28zzyjOF3AF1dJVCp+7rbh/hEF4Pu8WEIFEmc79EX89ldoB+vsFLStu6PBbhv//2yxuC2JHADr4C1o1ooB7lrmaT1+jRlK/y6Q7Pbr+7i6XEZpehi3fQUjzfLbDD09BIMDRU8HLaCrPE/7zHPvoruP8AuIONiCc1DDCSLmu+dbkA4ORXrqIv3uedfrD15uwXp6CNYr0FDodoRACyjRgfO7FvD1SmFFwE+wGylXeRkj7hffUVtc3hzo2/CjtrphZf5c65FG01r1oL1qqhVJ8biaBy2dbh2h9arfXjHWieP6uHr7zTFs5bZrvC49EgIBq+q7tfW/eALOsBl2AuD0c7IHMnvgW/73Z/2az8HsVMKeWtiBPxvUYA27CeazBU792WmtwNogB9bzucA2COInn9ewKaiX4zXM5NkpXeZcjurNXg8s5oG/Pt2WS7ziYMVAjINtWPsWa99fD7blEic4WGFoZ/0tWKW4RL7w13rkVY6Vl1oDCKM1e7Nv35S3OZkecp9qri8U8wfoFn/BuMs1S6U+tUhWep9Atzr6jRAFGgp/ukBfV9PJITUpoBt7Sxo/8LT9xOqEc++mtcoir3440XXetFXHEFzfzc5OvazVEM73Xa3RkcaKiMIeKaSgXoMUWPonvR/w6FSQGybP4rrK4rJsQh/38aBVUEz0kk5mfd2oYzzZ6r1AE4ep/CmnwaetQ98C2GIwtZc2opLCpS3W+VpdCNEHIa2ZSEvkcdYTPiLoJd/boMsHKvSyfbmhppV+md1SzSX2H2pRNmhiirvwwiJiZf98wVYFN1mhhiz17Ug0PY0zqeEYD3BBQYoBJqoPZHnYa+DlT6GTwwbNqt7C5oP9l6FW6tspcQChuFscQcj6x1THpMlJ2yx4mjjODl2SphZIx2lEm5TjzKARR+UglxPaA9MmTZ/rNs9yG7OQ3YEq2NezRxhJdb4Wjs+j8J1vjLzGqoM/YG+KIGBUvis5diDilo35/RE8k580UWgnftGmrsRrIlvexLxBVDes/T9rauzYlKtQftdRGVVVwctf1+vYHnaCnWELOx83n7AMWhAL2PYl6MNG4XWvY+/XY1gE9X3xnhvWapuc8F1jm+pM4Bx3IJH+n9pj1WMH0mIzedQ+3j6AQ+nrKklLijar4RHv2zyaxYWWHp/ZKsEF+Kffma5nUQcHUekt8BMjEkefo4T8NGEpI//2znS1VvQ/KECGtqc1rRa9Kve5dhEvUHOsPuNxuLjNhqqGVQRKJ/PhDixiAGs4hHVLPFyBb7yo4NYUbWI08jpZ1WFyYJ/rL12I3qREeNwVmFGwhgynw3bzbR/tGBnsVy0xbNQxPUeMtw4Eth0j8b0rrpZm4TJe5sVmtHWo5aETkmtnNwS9v7wTtXO0P70X6lYsGf7QVPLqHG+ivSWT4du81JEj8hUIJGigldCU0jubRcv/8F1rPwbcBn5Am548aUwL22PniRULIh9VscAy1Xld4hEm5isVrwMX5e1mZR4NjPCwqq45/u81CDIg5kGHK8wRpGryAtVJsnnHHFGmsXqRBieODQZOac5VPdNzBQ6qZlA7UeGQpmIF1NMKH01TjeVDLNtGJ+xcyetjadwz5Lw0Lt28vPBhNLfznj4eGkHbCacO2VBaEQYabZ6FZO5Ca52VRnL1L1XQhseUMxxJ9yBDdP7zUfBWeQgl45YSbAfXRxgqHnIdhEbPTAplO1e0QrP2TpO9hmvg7F1F0xuyUmzrRFnFq1qmHFJ9JMR8RShncRWB+hf0YOHxg0BYRZhD8KjMd4/Q9rIHvfSgFlJF09pAb2cgKw9r7INhaHcyz9cx0Ncw0YFs8T/FGu64PGqMi/y2eXuIyNsZMrVrt3osN9n7YSs+QJGM1rgNpNvXKGUb3ijD1Fch4JQq03ZHSAnXxuWbRrweBRgccEDHD2hphxldw7aIPdo4kd+ObU8n3l9cgd5yVipkfa2wa9Wsi31ngvMRWfQ5PjjOIfEo+8Ygu+F1Dp7tNsnfFpuPAxPh3Tzx1ko76A5GamMMhyItgf6aSWY9rp+t6G+u92eDlE9HXr8e+9kBUDfX3XfYms2ADf2NeAxjlfMLZjwyKM1821KYwasQ3cNxLzWSGjoPAcVOEuNgMY6b46aMBSaWBFaKfjDu+2++OkBuquay+pr7ki7cbZ9DNR8mTSHtbd40mtou8pH0vutn4ok53sWJ7RW7njgPvam9XzMedstGWu3RGuFYzd4ti+8qv6mFBmo7d9S6PB7qc7XPjrLdhmFqaDGIkGXD2jTqu7/dSzIeujcO/Hkh49YEwRjvlExqS0mP2yX9ukwsGcIJ80cw0tYgMoc7e2k3qdYHBnnPCLDSJaxtKiiniP22hm2mG8GGxHZEdSdy824Pr92PKrZ7ee1z79qNc/I4Dy8pUQy93Qcl9/p1if5k25CIyXeAYxZpgoghvDdo6u5ZB/hlVSMhXFokmLaFsiQkQVkZWfl4X62D/bRMq9AHy06busfWn+uR5awd+vujd8ZNn6iDlSzDD7sdovj5uH3YyjilaC4W7dc8J+cOiUsud1wMnR+cd+fc2x3EDS6WQNqUMZxsGimjt3viuC9QTNeL0tV1JN9Nfv6idUVUlZio3NPcSPAvDykzxQuYtTcnq1WHWsiHT7h+DAa0iMBUQvk/siF1RLAKtxCtGM9P8P/dxTKnGJZ4yYdo24o9HMkdX0uzt0bGaWnt9qA4LW3cHg+n/r7lTazOmlGA8qa0EZBUDecYs+kRW7+PIS+QGfCg0c1+yhl0AIkDNoYaJrvXY4VAvgspH+1FGa4C6z0mQIHzksL+WAUHGP3tWvkIqaWlhT14VOtp27vE4sUFvOpl7yXFoAXhBUhyxyB8j796BOOxXmYUSdfn0Psd9TxI0PX8zs/0upBXFevqukOkAQUzppAzzgVT79REgAwm9SAaFHzEDAEFIDHj0WSsQwIdiJabxzl0RY5EU1QwAQ2kDYLPwu7IbLNKq165vlqleVWiDROPaFUF+KBnsiCTt/8c7w47Wc8DZ1GUmXQr1O2Xqm2Mny6ZIhhZv0oqvJ5F8jFAOv0jWfkI29W9Der4eskSY1Kw6A7qmWRRRKSMupGPQ+Z38vlcPOKAL6GW6xUXm3fuVQiDb3Vwhm8nD7ADWhRxnI0+fUR+nI1ew+dVuo5Hr98Dl8zzrGJfp4GLtiJLOIwLTAq/ViNMcIw77K6z+sVfDmRcn//hPUyI4cnp7KG3yhYYqWOVjPrfSZcIGFZ+fQwvrOitIwyy0vPCab6EjVYCC6+vwvA0gu/suv7yRZPfxt0YUxymrgq91JXBlromslKXIikpIcAsW2dYw5/oEvJChDh2b6ipigOt1/AbGSe9l0BdvIaq+N4SIghqoie/pdDcBoCe4kOtS05gDrVNsblbQ+O4oXVUo+xsALa8igofoxDgReonOooso1WxSUcqkS7zjMiEpcKOKMuEYv67KpnelISM+/vE82t9pDh5nrkuLQvyVSF/vOytV+SWSu870EkhDAnv+pY9nikcWXwpu+Qg3KTJ0qcds5uKVfidS5hjccfJYrSdH9UFAc9cENBBTzn0p1ZFTVmcGT7J4s6mo9Sz0Ulv8MJApPECJWZwtGNeHjYnO41pJGfiSzUTW2fhLVqiQ36KOyyzBAiMDwn8syZhhQt/+5Gafh58wm+OZxRzgYgcUqleFd/x6Zpu7WPRqHKMyPj+rJY+BdmCfl9CL+mIggpwMCaPHkRVrzmNvFP3aVWp7oPMxon3rTPxvp0Me/eM4mH+4P3L77+vlv/i1SHLauYCqvbhjRWa0lat/P33q3iRZPdAk3UaFQ/3aUH/5L4YgKp8VazTWCa+I6bxvvHe5L/C359wl/iN99GE3/vGu+Bofd94r/AKo3deFD34cQZ6DH//Hf651SyTmajGqkzhoVKLetga4kjR81vmdVz+vvHuebC63rcgwb8N3CSU5pzWqSMyMEbS1wuz1N9SnNph4uCZcR3MCQfluhHqJMjvv/8fJlzNbsA9Vy9uj3Fc86rKl0xOpC2km1E2ll8zf3o8kys0XdqHmAkpHithUsvW98iYcNC91g4Nxp4L1WMNtCzyqgFyrJzGGaHgvXkvSfOpNgVpBNgIJxD4vPMjVsFD511tiNmGQAaEvJrf399jE8TYPjx48rlm2vxjmZECNE1Cop7O+U1F4GOdD+wgs3sdec9XiAtJNaf1OyjGbyyss5AcKWkvhy+gE6l9fuT060IxuBK3uVGaiBCwrL67W9h/P36HscxhuyntXsMtttV6aPwdpmvnr63vQ1WuH1kZBwp/VFVqX1yo93fGdVshvy8j6S92O8p6rN+02bsPmtSieXE8A59rh63BIgNFIWQjFd/uEUgEi8zXaQo7SD3U/r5TD4etakH3dD5demczdmu+9QJri9rXxmIt/nG7HxpRRxyNh6IC85ps2PDpte/O8DM4E/OMDFNVzRanLdYqLTcdX2xG4PdijS+5Q+sjJzoBMS3etLS/nNl2yPSvBQuxPdXLm6zJwrX3FFRffPLJb2tP4HTJsod1A9NJLZdI9ZGhCE0hR4oeP5p0WwYR3zua1PqIFe035OMF6/qlakm3rab7HTepRRcPvlHNrcVIpMpoPKWQn+pybe2+9590r7edE7gl+3hgTISb0HsRX0IUh02ElBDuX5u6Q1abN/uf6+dqbviYBjmnz+rAmdlKuBV/Na8K15HDBvdgn5BGr/ZPCZoWdLzV8pTk9smxfYI0j7MkfflMyxK59WyrMXPc2SN6NxbnJBPnHMkxdDXYXhV3cW7jfabOdu7/oh5bT5G7abyqvDfE8OQyj6dCkFq3iOIFp3lnfHmTrNhvfuIJ13o8tjXMPvTuAYF5GVW58IsVb6fD/vvcs4qDd1VEGVRxC43S4qqnEcs40hyfQKkrtqo2lcVRhqnk7mDYQaOyAyaNBVfJ2yaM2Hzsx8r7eL1/34Jxr4FRCIR9ZhABuuUMxF3525lYYNGqpaBmV9BAqJDccTzZnNODgaVfM7fw9ai6PqkMSPg00Ua89OH7fbwn/F3QOBdj9XpNLwRAYo7qzFZgDtPDvs1RSuINK5KniZhOhxMv9Rsemn/mwL4t4Y6x7i61QJnIyjKGxUcVYCPthKz5PY7QaHL4dF1l2W6H2+ui/C6DObU5BVvqrOU3a14mygdoS5WN/nEBUdWJqkpnKAO0zlP+VQWGbWipBsneUgsi/Ysupiv7C7W69mZN+8aNB8ewr8KqSnGIJ2wfgslHxUhcoqUJzczUKeutgxx9TBwN7SK3JrO7hnaWcRgVC8Xs8I2Seu9RmK0i7eZAiegrZQDOisUaJ8FHyvHRhbhIVuRr13mNUsP7WOR/R6bof2deuRJGXStyO4GoohfNZtgewu13jo/VlYxjfB8WhA/NDjYPcmjEEd80u47T1aiDryCCRuSd4cthKR3vqVtUuythCXg8S4rWKjqcX3Z0PR/4XR+Aj1HebHZjL2EJOzb7CbTtTJlOqNjEoOms4wMQgI75pUVxff6Csn+sk+lNezlNiF9hNa2SbKOuUNJ7XiXt/mfx1XqxgDHf0z7Nm7vrgRVbn6EIbs6zdONeolO1SLY2jh94AcMV6AhAty4FtFIg8FfPLAVCgWhfKGo26aNd64V88XhE9cgUE0cJmItz7W+3dTQ8Q+dtvcazubggPG+ACOc1AHjZRCGey20HEDEesYqT8OTkxIGSF2Kk82/LLSpE0KhhR2i2kdfs0R4/v3qRNrcDaoax7bTakxqKumOdoAcTmG9g0tl0F9xsFWvgkO4CWqW6BokZ5mLVUYKXqNHUFoZ0UBeGyOlh2Bkqxxu6d/R/AXJBsYs="""
    module_code = zlib.decompress(base64.b64decode(embedded)).decode('utf-8')
    (src_dir / '__init__.py').write_text('', encoding='utf-8')
    (src_dir / 'building_footprint_final.py').write_text(module_code, encoding='utf-8')
    print('Created embedded module at', src_dir / 'building_footprint_final.py')

sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from src.building_footprint_final import (
    ExperimentConfig,
    ensure_output_dirs,
    prepare_manifests,
    run_metric_self_tests,
    run_classical_baselines,
    tune_svm,
    predict_svm_patch,
    train_unet,
    predict_unet_patch,
    evaluate_method,
    aggregate_summary,
    plot_method_comparison,
    write_report_snippets,
    save_qualitative_grids,
)

import pandas as pd

## 1. Configuration

Các giới hạn patch mặc định được đặt để chạy được trên Kaggle GPU trong thời gian hợp lý. Khi cần full experiment hơn, tăng `max_train_patches`, `max_val_patches`, `max_test_patches`.

In [ ]:
cfg = ExperimentConfig(
    output_dir='/kaggle/working/outputs' if Path('/kaggle/working').exists() else 'outputs',
    dataset_root=None,
    random_seed=42,
    max_train_patches=1200,
    max_val_patches=300,
    max_test_patches=300,
    unet_epochs=20,
    unet_batch_size=4,
    qualitative_examples=8,
)

RUN_CLASSICAL = True
RUN_SVM = True
RUN_UNET = True

# Debug nhanh khi cần test notebook trước khi train thật.
QUICK_DEBUG = False
if QUICK_DEBUG:
    cfg.max_train_patches = 24
    cfg.max_val_patches = 8
    cfg.max_test_patches = 8
    cfg.max_train_pixels = 20_000
    cfg.unet_epochs = 1
    cfg.unet_batch_size = 2
    cfg.max_svm_val_patches_for_tuning = 4
    cfg.max_unet_val_patches_for_threshold = 4
    cfg.qualitative_examples = 2

paths = ensure_output_dirs(cfg)
print(paths)

## 2. Sanity Tests

Cell này kiểm tra metric edge cases: perfect mask, empty mask, missed prediction và morphology shape.

In [ ]:
run_metric_self_tests()

## 3. Dataset Manifest

Split được tạo ở cấp ảnh gốc, sau đó mới sinh patch manifest. File quan trọng: `outputs/manifests/split_manifest.csv`.

In [ ]:
paths, records_by_split = prepare_manifests(cfg)
for split, records in records_by_split.items():
    print(split, len(records), 'patches')

pd.read_csv(paths['manifests'] / 'image_split_manifest.csv').head()

## 4. Classical Baselines: K-Means và Otsu

Hai baseline này dùng lại logic giữa kỳ, nhưng chạy lại trên đúng `test_manifest` của bản cuối kỳ.

In [ ]:
all_frames = []
qualitative_predictors = {}
test_records = records_by_split['test']

if RUN_CLASSICAL:
    classical_df, classical_predictors = run_classical_baselines(test_records, cfg)
    all_frames.append(classical_df)
    qualitative_predictors.update(classical_predictors)
    display(aggregate_summary(classical_df))

## 5. Supervised ML Baseline: Linear SVM

SVM dùng feature pixel-level: `RGB`, `LAB`, `HSV`, grayscale, Sobel gradient magnitude, local mean và local std. Hyperparameter `C` được chọn theo Dice trên validation.

In [ ]:
if RUN_SVM:
    svm_model, svm_tuning_df, best_c = tune_svm(records_by_split['train'], records_by_split['val'], cfg)
    print('Best SVM C =', best_c)
    display(svm_tuning_df)

    svm_df = evaluate_method(
        'SVM',
        test_records,
        lambda patch: predict_svm_patch(patch, svm_model, cfg),
        cfg,
    )
    svm_df['selected_c'] = best_c
    all_frames.append(svm_df)
    qualitative_predictors['SVM'] = lambda patch: predict_svm_patch(patch, svm_model, cfg)
    display(aggregate_summary(svm_df))

## 6. Deep Learning: U-Net

U-Net dùng `BCEWithLogitsLoss + Dice loss`, `AdamW`, early stopping và threshold tuning trên validation. Checkpoint tốt nhất được lưu tại `outputs/models/unet_best.pth`.

In [ ]:
if RUN_UNET:
    import torch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device =', device)

    unet_model, unet_threshold, unet_history_df = train_unet(
        records_by_split['train'],
        records_by_split['val'],
        cfg,
    )
    print('Selected U-Net threshold =', unet_threshold)
    display(unet_history_df.tail())

    unet_df = evaluate_method(
        'U-Net',
        test_records,
        lambda patch: predict_unet_patch(patch, unet_model, cfg, unet_threshold, device),
        cfg,
    )
    unet_df['threshold'] = unet_threshold
    all_frames.append(unet_df)
    qualitative_predictors['U-Net'] = lambda patch: predict_unet_patch(patch, unet_model, cfg, unet_threshold, device)
    display(aggregate_summary(unet_df))

## 7. Final Metrics and Artifacts

Cell này gom toàn bộ kết quả thành output contract cuối kỳ.

In [ ]:
if not all_frames:
    raise RuntimeError('No experiment branch was run.')

per_patch_df = pd.concat(all_frames, ignore_index=True)
summary_df = aggregate_summary(per_patch_df)

per_patch_df.to_csv(paths['metrics'] / 'per_patch_metrics.csv', index=False)
summary_df.to_csv(paths['metrics'] / 'final_summary.csv', index=False)
plot_method_comparison(summary_df, cfg)
write_report_snippets(summary_df, cfg)
save_qualitative_grids(test_records, qualitative_predictors, cfg)

display(summary_df)
print('Saved final_summary:', paths['metrics'] / 'final_summary.csv')
print('Saved per_patch_metrics:', paths['metrics'] / 'per_patch_metrics.csv')
print('Saved figures:', paths['figures'])
print('Saved report snippets:', paths['reports'])

## 8. Error Analysis Helpers

Bảng dưới giúp chọn case tốt/xấu cho phần phân tích lỗi: shadow, roof color, small roofs, parking/road confusion, false merge.

In [ ]:
error_view = per_patch_df.sort_values(['method', 'dice'])[
    ['method', 'patch_id', 'city', 'iou', 'dice', 'precision', 'recall', 'count_err', 'area_abs_error']
]
print('Worst cases by Dice')
display(error_view.groupby('method').head(5))

print('Best cases by Dice')
display(per_patch_df.sort_values(['method', 'dice'], ascending=[True, False]).groupby('method').head(5)[
    ['method', 'patch_id', 'city', 'iou', 'dice', 'precision', 'recall', 'count_err', 'area_abs_error']
])

## 9. Files for Submission

Sau khi chạy xong, các file cần lấy từ Kaggle output:
- `outputs/metrics/final_summary.csv`
- `outputs/metrics/per_patch_metrics.csv`
- `outputs/figures/method_comparison.png`
- `outputs/figures/qualitative_grid_*.png`
- `outputs/models/linear_svm_building_footprint.joblib`
- `outputs/models/unet_best.pth`
- `outputs/reports/final_metrics_table.tex`